In [1]:
import torch
import torch.nn as nn

In [2]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.projector = nn.Linear(3, 16)
        self.ffn = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, runs, wickets, balls, labels):
        x = torch.stack([runs, wickets, balls], dim=1).float()
        x = self.projector(x)
        x = self.ffn(x)

        logits = x.squeeze(1)
        loss = self.loss_fn(logits, labels.float())
        return {"loss": loss, "logits": logits}


In [3]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").is_dir():
    pass
elif (project_root / "cricket-win-predict" / "src").is_dir():
    project_root = project_root / "cricket-win-predict"
elif project_root.name == "notebooks" and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
else:
    raise FileNotFoundError("Could not locate the cricket-win-predict project root")

sys.path.insert(0, str(project_root))

In [4]:
from src.data.filter import load_data, filter_with_nation_winners
from src.data.preprocess import process_data

data = load_data()
filtered_data = filter_with_nation_winners(data)
processed_data = process_data(filtered_data)[1]

In [5]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(processed_data, test_size=0.2, random_state=42)

In [6]:
from transformers import Trainer, TrainingArguments

train_args = TrainingArguments(
    per_device_train_batch_size=32,
    per_device_eval_batch_size=5000,
    num_train_epochs=1,
    learning_rate=1e-3,
    output_dir=str(project_root / "models" / "simple_model"),
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=100,
)

model = SimpleModel()

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

trainer.train()

/home/rohan/TimePass/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step,Training Loss,Validation Loss
100,0.532766,0.473931
200,0.484290,0.444214
300,0.463874,0.432002
400,0.476396,0.431254
500,0.443947,0.415271
600,0.437404,0.422062
700,0.420110,0.457263
800,0.411776,0.406269
900,0.414415,0.404174
1000,0.402605,0.397586


TrainOutput(global_step=13120, training_loss=0.38864106873913506, metrics={'train_runtime': 129.7987, 'train_samples_per_second': 3234.484, 'train_steps_per_second': 101.08, 'total_flos': 0.0, 'train_loss': 0.38864106873913506, 'epoch': 1.0})